# LoRA: Training Apple's 3B Model on A100

Standard LoRA training using Apple's adapter toolkit. Requires A100 (40GB GPU).
For free T4 training, see `train_qlora.ipynb`.

**Results:** 3 epochs, ~2.5 hours. Loss: 1.94 → 1.17 → 0.61 (train), eval: 1.50 → 1.12 → 1.10

## 1. Setup

Upload to `My Drive/hunch-training/`:
- `adapter_training_toolkit_v26_0_0/` (from developer.apple.com)
- `prepare_data.py`, `tldr_bank.db`, `prompts.jsonl`

In [3]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/hunch-training'
WORK_DIR = '/content/hunch-training'

!mkdir -p {WORK_DIR}
!cp -r {DRIVE_DIR}/adapter_training_toolkit_v26_0_0 {WORK_DIR}/
!cp {DRIVE_DIR}/prepare_data.py {WORK_DIR}/
!mkdir -p {WORK_DIR}/../bank {WORK_DIR}/../benchmark
!cp {DRIVE_DIR}/tldr_bank.db {WORK_DIR}/../bank/
!cp {DRIVE_DIR}/prompts.jsonl {WORK_DIR}/../benchmark/

!cd {WORK_DIR}/adapter_training_toolkit_v26_0_0 && pip install -r requirements.txt -q

import torch
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 117.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 362.6/362.6 kB 40.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 93.0 MB/s eta 0:00:00
CUDA: True
GPU: NVIDIA A100-SXM4-40GB


## 2. Prepare training data

In [4]:
!cd {WORK_DIR} && python3 prepare_data.py --sources override

Loaded 21478 entries from bank
Filtered to sources {'override'}: 134 entries (from 21478)
  override: 134
Excluded 38 entries matching benchmark prompts
After dedup: 96 unique entries (removed 0)
Small dataset — using all 96 examples for both train and eval
Wrote 96 examples to /content/hunch-training/train.jsonl
Wrote 96 examples to /content/hunch-training/eval.jsonl

Sample training examples:
  user: show response headers
  asst: curl -I https://example.com

  user: dns lookup for a domain
  asst: dig example.com

  user: record shell session to file
  asst: script session.log



## 3. Train

No patches needed for A100. ~25 min/epoch, ~1.5 hours total.

**Note:** lr=1e-3 diverged in testing. Use 1e-4.

In [5]:
!cd {WORK_DIR}/adapter_training_toolkit_v26_0_0 && python3 -m examples.train_adapter \
  --train-data ../train.jsonl \
  --eval-data ../eval.jsonl \
  --epochs 20 \
  --learning-rate 1e-4 \
  --batch-size 8 \
  --precision bf16-mixed \
  --activation-checkpointing \
  --checkpoint-dir ../lora-override-checkpoints/

Fine-tuning adapters with configuration: 
AdapterTrainingConfiguration(epochs=20, learning_rate=0.0001, batch_size=8, linear_warmup_epochs=1, gradient_accumulation_steps=1, enable_activation_checkpointing=True, precision='bf16-mixed', compile_model=False, weight_decay=0.01, clip_grad_norm=1.0, max_sequence_length=None, fixed_sized_sequences=False, pack_sequences=False, loss_update_frequency=3)
Loading base model on cuda with precision torch.float32
/usr/local/lib/python3.12/dist-packages/tamm/layers/flash_attention.py:78: UserWarning: Failed to import flash-attn for Flash attention. Using flash attention may lead to significantly faster training. Please refer to tamm-scripts/install_flash_attn.sh for instructions.
  _warnings.warn(
Total parameters 3178001792
Total trainable parameters 66633728
Gradient scaling is enabled: False
Epoch 1/20
Training: 100% 12/12 [00:08<00:00,  1.42it/s, loss=1.64]
Evaluation: 100% 12/12 [00:02<00:00,  5.12it/s, loss=1.05]
Epoch 2/20
INFO:examples.utils:E

## 4. Save checkpoints

In [6]:
!cp -r {WORK_DIR}/lora-override-checkpoints {DRIVE_DIR}/lora-override-checkpoints
!echo 'Checkpoints saved to Drive'

Checkpoints saved to Drive


## 5. Export .fmadapter

In [7]:
!cd {WORK_DIR}/adapter_training_toolkit_v26_0_0 && python3 -m export.export_fmadapter \
  --adapter-name hunch \
  --checkpoint ../lora-override-checkpoints/adapter-final.pt \
  --output-dir ../lora-override-exports/

!ls -lh {WORK_DIR}/lora-override-exports/
!cp -r {WORK_DIR}/lora-override-exports {DRIVE_DIR}/lora-override-exports
!echo 'Adapter exported and saved to Drive'

scikit-learn version 1.6.1 is not supported. Minimum required version: 0.17. Maximum required version: 1.5.1. Disabling scikit-learn conversion API.
XGBoost version 3.2.0 has not been tested with coremltools. You may run into unexpected errors. XGBoost 1.4.2 is the most recent version that has been tested.
2026-04-15 17:21:10.930166: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-15 17:21:10.949095: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776273670.972532    4269 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1

## 6. Evaluate

In [8]:
import json, subprocess

test_prompts = [
    'find files changed in the last hour',
    'show disk usage',
    'generate a random password',
    'kill a process by name',
    'show http headers of a url',
    'record terminal session',
    'find files larger than 100mb',
    'convert image to different format',
    'show all listening ports',
    'find files modified in the last 7 days',
    'find files owned by root',
    'count lines in all python files',
    'show all environment variables',
    'clear the terminal',
    'compare two files',
]

system = 'Output a single shell command for zsh on macOS. No explanation, no markdown, no backticks. Just the command.'

with open(f'{WORK_DIR}/test_prompts.jsonl', 'w') as f:
    for p in test_prompts:
        f.write(json.dumps([
            {'role': 'system', 'content': system},
            {'role': 'user', 'content': p}
        ]) + '\n')

result = subprocess.run(
    ['python3', '-m', 'examples.generate',
     '--prompt', '../test_prompts.jsonl',
     '--checkpoint', '../checkpoints/adapter-final.pt',
     '--precision', 'bf16-mixed'],
    capture_output=True, text=True,
    cwd=f'{WORK_DIR}/adapter_training_toolkit_v26_0_0'
)

lines = (result.stdout + result.stderr).strip().split('\n')
idx = 0
for line in lines:
    if 'Response for prompt' in line:
        answer = line.split(': ', 2)[-1].replace('<turn_end>', '').strip()
        prompt = test_prompts[idx] if idx < len(test_prompts) else '?'
        print(f'Q: {prompt:<45} A: {answer}')
        idx += 1

if idx == 0:
    print('No output. Check error:')
    print('STDERR:', result.stderr[-500:])
    print('Return code:', result.returncode)

No output. Check error:
STDERR: ^^^^^^^
  File "/content/hunch-training/adapter_training_toolkit_v26_0_0/examples/utils.py", line 167, in load_base_model
    with Path(checkpoint_path).open("rb") as f:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/pathlib.py", line 1013, in open
    return io.open(self, mode, buffering, encoding, errors, newline)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: '../checkpoints/adapter-final.pt'

Return code: 1
